# Metadata-Based AI Image Detection Module

## 1. Imports

In [1]:
from __future__ import annotations

import json
import re
import subprocess
import sys
from pathlib import Path
from typing import Any

from pydantic import BaseModel, Field

## 2. Configuration

In [2]:
AI_KEYWORDS = [
    "openai",
    "dall-e",
    "dalle",
    "chatgpt",
    "stable diffusion",
    "stability ai",
    "midjourney",
    "firefly",
    "adobe firefly",
    "ideogram",
    "flux",
    "runway",
    "leonardo",
    "imagen",
    "grok",
    "generated by ai",
    "ai generated",
    "synthetic",
    "claim_generator",
    "created_software_agent",
    "content credentials",
    "c2pa",
]

CAMERA_KEYS = [
    "make",
    "model",
    "lensmodel",
    "lensinfo",
    "focallength",
    "aperturevalue",
    "exposuretime",
    "isospeedratings",
    "makernote",
]

SOFTWARE_KEYS = [
    "software",
    "processingsoftware",
    "creatortool",
    "historysoftwareagent",
    "claim_generator",
    "created_software_agent",
]

C2PA_KEYS = [
    "c2pa",
    "manifest",
    "contentcredentials",
    "claim_generator",
    "created_software_agent",
]

BINARY_MARKERS = [
    b"c2pa",
    b"claim_generator",
    b"created_software_agent",
    b"openai",
    b"dall-e",
    b"dalle",
    b"midjourney",
    b"stable diffusion",
    b"firefly",
    b"ideogram",
    b"flux",
]

## 3. Output Models

In [3]:
class FeatureSet(BaseModel):
    has_make: bool = False
    has_model: bool = False
    has_lens_model: bool = False
    has_makernote: bool = False
    has_gps: bool = False
    has_c2pa: bool = False
    has_ai_claim: bool = False
    has_camera_claim: bool = False
    has_edit_claim: bool = False
    software_tag_count: int = 0
    xmp_namespace_count: int = 0
    keyword_hits: list[str] = Field(default_factory=list)
    binary_hits: list[str] = Field(default_factory=list)
    suspicious_only_software_tags: bool = False
    suspicious_perfect_timestamp: bool = False


class AnalysisResult(BaseModel):
    file: str
    metadata_extracted: bool
    metadata_summary: dict[str, Any]
    features: FeatureSet
    ai_probability: float
    decision: str
    rationale: list[str]

## 4. Metadata extraction utilities

In [4]:
def run_exiftool(image_path: Path) -> dict[str, Any]:
    """
    Extract as much metadata as possible using ExifTool.
    -j returns JSON
    -a allows duplicate tags
    -u keeps unknown tags
    -g1 groups tags by family 1
    """
    cmd = [
        "exiftool",
        "-j",
        "-a",
        "-u",
        "-g1",
        str(image_path),
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        check=False
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or "ExifTool failed")

    data = json.loads(result.stdout)
    if not data:
        return {}
    return data[0]

In [5]:
def flatten_metadata(data: Any, prefix: str = "") -> dict[str, str]:
    """
    Flatten nested JSON-like metadata into lowercase key/value pairs.
    """
    flat: dict[str, str] = {}

    if isinstance(data, dict):
        for key, value in data.items():
            new_prefix = f"{prefix}.{key}" if prefix else str(key)
            flat.update(flatten_metadata(value, new_prefix))
    elif isinstance(data, list):
        for i, value in enumerate(data):
            new_prefix = f"{prefix}[{i}]"
            flat.update(flatten_metadata(value, new_prefix))
    else:
        flat[prefix.lower()] = str(data).lower()

    return flat

In [6]:
def scan_binary_markers(image_path: Path) -> list[str]:
    """
    Lightweight binary scan for marker strings.
    Useful when metadata parsers miss embedded text or non-standard blocks.
    """
    try:
        blob = image_path.read_bytes().lower()
    except Exception:
        return []

    hits = []
    for marker in BINARY_MARKERS:
        if marker in blob:
            hits.append(marker.decode("utf-8", errors="ignore"))
    return sorted(set(hits))

## 5. Heuristics and scoring

In [7]:
def find_keyword_hits(flat: dict[str, str]) -> list[str]:
    haystack = " ".join([f"{k} {v}" for k, v in flat.items()])
    hits = [kw for kw in AI_KEYWORDS if kw in haystack]
    return sorted(set(hits))

In [8]:
def count_xmp_namespaces(flat: dict[str, str]) -> int:
    namespaces = set()
    for key in flat:
        if "xmp" in key:
            first = key.split(".")[0]
            namespaces.add(first)
    return len(namespaces)

In [9]:
def looks_like_perfect_timestamp(flat: dict[str, str]) -> bool:
    """
    Very rough heuristic: flags timestamps like HH:00:00.
    """
    patterns = [
        r"\b\d{4}:\d{2}:\d{2} \d{2}:00:00\b",
        r"\b\d{4}-\d{2}-\d{2}t\d{2}:00:00\b",
    ]
    for value in flat.values():
        for pattern in patterns:
            if re.search(pattern, value):
                return True
    return False

In [10]:
def build_features(flat: dict[str, str], binary_hits: list[str]) -> FeatureSet:
    keys = set(flat.keys())
    values = " ".join(flat.values())

    has_make = any("make" in k and flat[k].strip() for k in keys)
    has_model = any(
        ("model" in k and "lens" not in k and flat[k].strip())
        for k in keys
    )
    has_lens_model = any("lensmodel" in k or "lensinfo" in k for k in keys)
    has_makernote = any("makernote" in k for k in keys)
    has_gps = any("gps" in k for k in keys)

    has_c2pa = any(any(tag in k for tag in C2PA_KEYS) for k in keys) or ("c2pa" in values)
    has_ai_claim = (
        "generated by ai" in values
        or "ai generated" in values
        or "claim_generator" in values
        or "created_software_agent" in values
        or any(x in values for x in ["openai", "midjourney", "stable diffusion", "firefly", "ideogram", "flux"])
    )
    has_camera_claim = "captured" in values or "camera" in values
    has_edit_claim = "edited" in values or "photoshop" in values or "lightroom" in values

    software_tag_count = sum(
        1 for k in keys if any(tag in k for tag in SOFTWARE_KEYS)
    )

    suspicious_only_software_tags = (
        software_tag_count > 0
        and not has_make
        and not has_model
        and not has_lens_model
        and not has_makernote
    )

    return FeatureSet(
        has_make=has_make,
        has_model=has_model,
        has_lens_model=has_lens_model,
        has_makernote=has_makernote,
        has_gps=has_gps,
        has_c2pa=has_c2pa,
        has_ai_claim=has_ai_claim,
        has_camera_claim=has_camera_claim,
        has_edit_claim=has_edit_claim,
        software_tag_count=software_tag_count,
        xmp_namespace_count=count_xmp_namespaces(flat),
        keyword_hits=find_keyword_hits(flat),
        binary_hits=binary_hits,
        suspicious_only_software_tags=suspicious_only_software_tags,
        suspicious_perfect_timestamp=looks_like_perfect_timestamp(flat),
    )

In [11]:
def score_features(features: FeatureSet) -> tuple[float, list[str]]:
    """
    Conservative heuristic score for module 1.
    Important: weak/missing metadata -> uncertain, not 'real'.
    """
    score = 0.50
    rationale: list[str] = []

    # Strong positive evidence
    if features.has_ai_claim:
        score += 0.35
        rationale.append("Explicit AI-related generator or provenance signal found.")

    if features.has_c2pa:
        score += 0.10
        rationale.append("C2PA/content provenance-style metadata detected.")

    if features.keyword_hits:
        score += min(0.20, 0.03 * len(features.keyword_hits))
        rationale.append(f"AI keyword hits: {', '.join(features.keyword_hits)}")

    if features.binary_hits:
        score += min(0.15, 0.03 * len(features.binary_hits))
        rationale.append(f"Binary marker hits: {', '.join(features.binary_hits)}")

    if features.suspicious_only_software_tags:
        score += 0.10
        rationale.append("Software tags present without normal camera metadata.")

    if features.suspicious_perfect_timestamp:
        score += 0.05
        rationale.append("Timestamp pattern looks unusually clean/perfect.")

    # Camera-origin evidence reduces AI suspicion slightly, but not too much
    camera_evidence_count = sum([
        features.has_make,
        features.has_model,
        features.has_lens_model,
        features.has_makernote,
    ])

    if camera_evidence_count >= 3:
        score -= 0.20
        rationale.append("Rich camera metadata present.")

    elif camera_evidence_count == 2:
        score -= 0.10
        rationale.append("Some camera metadata present.")

    # Clamp
    score = max(0.01, min(0.99, score))

    return score, rationale

In [12]:
def decide(prob: float, features: FeatureSet) -> str:
    if features.has_ai_claim and prob >= 0.80:
        return "likely_ai_generated"
    if prob >= 0.75:
        return "likely_ai_generated"
    if prob <= 0.35:
        return "likely_camera_origin"
    return "uncertain"

## 6. Main analysis function

In [13]:
def analyse_image(image_path: str) -> AnalysisResult:
    path = Path(image_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {image_path}")

    try:
        raw_metadata = run_exiftool(path)
        metadata_extracted = True
    except Exception as exc:
        raw_metadata = {"error": str(exc)}
        metadata_extracted = False

    flat = flatten_metadata(raw_metadata)
    binary_hits = scan_binary_markers(path)
    features = build_features(flat, binary_hits)
    ai_probability, rationale = score_features(features)
    decision = decide(ai_probability, features)

    summary = {
        "top_level_keys": sorted(list(raw_metadata.keys()))[:20] if isinstance(raw_metadata, dict) else [],
        "keyword_hits": features.keyword_hits,
        "binary_hits": features.binary_hits,
    }

    return AnalysisResult(
        file=str(path),
        metadata_extracted=metadata_extracted,
        metadata_summary=summary,
        features=features,
        ai_probability=round(ai_probability, 4),
        decision=decision,
        rationale=rationale,
    )

## 7. Run on one image

Change the path below to your own test image.

In [14]:
image_path = "sample_images/test_1.jpg"

result = analyse_image(image_path)
result.model_dump()

{'file': 'sample_images/test_1.jpg',
 'metadata_extracted': True,
 'metadata_summary': {'top_level_keys': ['Composite',
   'ExifTool',
   'File',
   'ICC-header',
   'ICC_Profile',
   'IFD0',
   'IPTC',
   'JFIF',
   'SourceFile',
   'System'],
  'keyword_hits': [],
  'binary_hits': []},
 'features': {'has_make': False,
  'has_model': True,
  'has_lens_model': False,
  'has_makernote': False,
  'has_gps': False,
  'has_c2pa': False,
  'has_ai_claim': False,
  'has_camera_claim': False,
  'has_edit_claim': False,
  'software_tag_count': 0,
  'xmp_namespace_count': 0,
  'keyword_hits': [],
  'binary_hits': [],
  'suspicious_only_software_tags': False,
  'suspicious_perfect_timestamp': False},
 'ai_probability': 0.5,
 'decision': 'uncertain',
 'rationale': []}

In [15]:
image_path = "sample_images/test_2.jpg"

result = analyse_image(image_path)
result.model_dump()

{'file': 'sample_images/test_2.jpg',
 'metadata_extracted': True,
 'metadata_summary': {'top_level_keys': ['Adobe',
   'Composite',
   'ExifIFD',
   'ExifTool',
   'File',
   'ICC-header',
   'ICC-meas',
   'ICC-view',
   'ICC_Profile',
   'IFD0',
   'IFD1',
   'IPTC',
   'Photoshop',
   'SourceFile',
   'System',
   'XMP-aux',
   'XMP-crs',
   'XMP-dc',
   'XMP-iptcCore',
   'XMP-photomech'],
  'keyword_hits': ['imagen'],
  'binary_hits': []},
 'features': {'has_make': True,
  'has_model': True,
  'has_lens_model': True,
  'has_makernote': False,
  'has_gps': False,
  'has_c2pa': False,
  'has_ai_claim': False,
  'has_camera_claim': True,
  'has_edit_claim': True,
  'software_tag_count': 3,
  'xmp_namespace_count': 11,
  'keyword_hits': ['imagen'],
  'binary_hits': [],
  'suspicious_only_software_tags': False,
  'suspicious_perfect_timestamp': False},
 'ai_probability': 0.33,
 'decision': 'likely_camera_origin',
 'rationale': ['AI keyword hits: imagen', 'Rich camera metadata present.'

In [23]:
image_path = "sample_images/test_3.png"

result = analyse_image(image_path)
result.model_dump()
print("ai_probability: ", result.ai_probability)

ai_probability:  0.88


## 8. Pretty JSON output

In [17]:
print(result.model_dump_json(indent=2))

{
  "file": "sample_images/test_3.png",
  "metadata_extracted": true,
  "metadata_summary": {
    "top_level_keys": [
      "CBOR",
      "Composite",
      "ExifTool",
      "File",
      "JUMBF",
      "Jpeg2000",
      "PNG",
      "SourceFile",
      "System"
    ],
    "keyword_hits": [
      "c2pa",
      "chatgpt",
      "claim_generator"
    ],
    "binary_hits": [
      "c2pa",
      "claim_generator",
      "openai"
    ]
  },
  "features": {
    "has_make": false,
    "has_model": false,
    "has_lens_model": false,
    "has_makernote": false,
    "has_gps": false,
    "has_c2pa": true,
    "has_ai_claim": false,
    "has_camera_claim": false,
    "has_edit_claim": false,
    "software_tag_count": 3,
    "xmp_namespace_count": 0,
    "keyword_hits": [
      "c2pa",
      "chatgpt",
      "claim_generator"
    ],
    "binary_hits": [
      "c2pa",
      "claim_generator",
      "openai"
    ],
    "suspicious_only_software_tags": true,
    "suspicious_perfect_timestamp": fals

## 9. Batch test a folder

This is useful for checking multiple images at once.

In [18]:
def analyse_folder(folder_path: str, extensions: tuple[str, ...] = (".jpg", ".jpeg", ".png", ".webp")):
    folder = Path(folder_path)
    results = []
    for file in sorted(folder.iterdir()):
        if file.suffix.lower() in extensions:
            try:
                results.append(analyse_image(str(file)).model_dump())
            except Exception as exc:
                results.append({"file": str(file), "error": str(exc)})
    return results

In [19]:
batch_results = analyse_folder("sample_images")
batch_results

[{'file': 'sample_images/test_1.jpg',
  'metadata_extracted': True,
  'metadata_summary': {'top_level_keys': ['Composite',
    'ExifTool',
    'File',
    'ICC-header',
    'ICC_Profile',
    'IFD0',
    'IPTC',
    'JFIF',
    'SourceFile',
    'System'],
   'keyword_hits': [],
   'binary_hits': []},
  'features': {'has_make': False,
   'has_model': True,
   'has_lens_model': False,
   'has_makernote': False,
   'has_gps': False,
   'has_c2pa': False,
   'has_ai_claim': False,
   'has_camera_claim': False,
   'has_edit_claim': False,
   'software_tag_count': 0,
   'xmp_namespace_count': 0,
   'keyword_hits': [],
   'binary_hits': [],
   'suspicious_only_software_tags': False,
   'suspicious_perfect_timestamp': False},
  'ai_probability': 0.5,
  'decision': 'uncertain',
  'rationale': []},
 {'file': 'sample_images/test_2.jpg',
  'metadata_extracted': True,
  'metadata_summary': {'top_level_keys': ['Adobe',
    'Composite',
    'ExifIFD',
    'ExifTool',
    'File',
    'ICC-header',
  

In [20]:
from pathlib import Path
import shutil

print("Working directory:", Path.cwd())
print("ExifTool path from Python:", shutil.which("exiftool"))
print("Image exists:", Path("sample_images/test_3.png").exists())
print("Resolved image path:", Path("sample_images/test_3.png").resolve())

Working directory: /Users/davidscerri/Library/Mobile Documents/com~apple~CloudDocs/Studies/Masters /Proof-of-Concept/Seeing-Through-Deepfakes/src/metadata_module
ExifTool path from Python: /opt/homebrew/bin/exiftool
Image exists: True
Resolved image path: /Users/davidscerri/Library/Mobile Documents/com~apple~CloudDocs/Studies/Masters /Proof-of-Concept/Seeing-Through-Deepfakes/src/metadata_module/sample_images/test_3.png


In [21]:
try:
    raw = run_exiftool(Path("sample_images/test_3.png"))
    print("ExifTool succeeded")
    print(raw.keys() if isinstance(raw, dict) else raw)
except Exception as e:
    print("ExifTool failed with error:")
    print(repr(e))

ExifTool succeeded
dict_keys(['SourceFile', 'ExifTool', 'System', 'File', 'PNG', 'JUMBF', 'CBOR', 'Jpeg2000', 'Composite'])
